# framediff: transforms and their code

A transform is an edit from one 64×64 frame toward another, and it exists twice:

- `transform(frame)` performs it here and returns the new frame;
- `to_code()` writes it as numpy — the block the model reads and the harness runs — and checks that text against `transform` before returning it;
- `bit_cost()` prices it, so a search can prefer the edit a person would name.

Every block has the same shape: copy the start frame `a` into a board `g`, edit `g`, and end with `assert (g == b).all()`. Rectangles are half-open and spelled exactly as the slices they emit.

A block may call two helpers, `from_ascii_grid` and `bresenham_line`. The REPL preloads both — it advertises `from_ascii_grid` beside `ascii_grid()`, and `bresenham_line` only through its `help()` — and `block_namespace()` binds framediff's own everywhere else.

`show(a, transform, b)` draws `a`, the result with the cells the transform changed outlined, and `b` (when given) with the cells the result still gets wrong outlined; below it the block and its cost.

In [ ]:
from pathlib import Path

from framediff import AsciiGrid, BlockCopy, Composite, FillRect, Line, NoOp, Recolour, SetPixel, from_ascii_grid
from framediff.diagnostics import show

frames = {}
for block in Path("tu93_frames.txt").read_text().strip().split("\n\n"):
    name, _, grid = block.partition("\n")
    frames[name] = from_ascii_grid(grid)

start = frames["level1_start"]
list(frames)

The frames are tu93's, from a recorded game: level 1 as it opens and after the first `RIGHT`, and a turn on level 2 before and after the `RIGHT` that lost it. Level 1 is an 11×11 lattice of 3×3 tiles; the blue tile at the top left is the player, its charcoal pixel the side it faces; the magenta bottom row is the energy bar.

## Nothing changed: `NoOp` (cost 0)

When the frames are equal, the block is the comparison alone: no board is built.

In [ ]:
show(start, NoOp(), start)

## One cell: `SetPixel` (cost 4)

In [ ]:
show(start, SetPixel(63, 63, 0))

## A rectangle of one colour: `FillRect` (cost 6)

Whatever its size, a fill costs the same.

In [ ]:
show(start, FillRect(15, 18, 15, 18, color=0))

## One colour for another inside a box: `Recolour` (cost 10)

Only cells of colour `old` inside the rectangle change. The block is a masked assignment into the rectangle's view of the board.

In [ ]:
show(start, Recolour(15, 48, 15, 48, old=2, new=14))

## A block copied: `BlockCopy` (cost 10)

The rectangle is filled from a same-sized block of `a`. A plain copy annotates its shift. It does not clear where it read from.

In [ ]:
show(start, BlockCopy(15, 18, 21, 24, src_row=15, src_col=15))

### A move: leaving a colour behind (still cost 10)

With `clear` set the copy is a move: the block's old place takes that colour, at no extra cost. The old place is cleared before the new one is written, so a move by less than the block's size comes out right.

In [ ]:
show(start, BlockCopy(15, 18, 21, 24, src_row=15, src_col=15, clear=0))

### Turned or mirrored (cost 18)

`flip` is `np.fliplr`, applied first; `rot` counts `np.rot90` quarter turns counterclockwise. Turning the player tile one quarter turn moves its charcoal pixel from the right side to the top; mirroring moves it to the left.

In [ ]:
show(start, BlockCopy(15, 18, 21, 24, src_row=15, src_col=15, rot=1))

In [ ]:
show(start, BlockCopy(15, 18, 21, 24, src_row=15, src_col=15, flip=True))

With an odd `rot` the block read is the rectangle's shape transposed: a 3×5 rectangle reads a 5×3 block.

In [ ]:
show(start, BlockCopy(30, 33, 50, 55, src_row=15, src_col=39, rot=1))

### From the board instead of the start frame

`source="g"` reads the board as the edits before it left it, so a copy can pick up what an earlier line wrote. Here the fill blanks a tile and the copy from `g` carries the blank along; the same copy from `a` would carry the original tile.

In [ ]:
show(
    start,
    Composite(
        (
            FillRect(15, 18, 15, 18, color=11),
            BlockCopy(21, 24, 21, 24, src_row=15, src_col=15, source="g"),
        )
    ),
)

## A straight line: `Line` (cost 12)

The block calls `bresenham_line`, the integer Bresenham the ARC games hand-roll (r11l's, checked cell for cell in the tests), endpoints included. The REPL preloads it without advertising it, so the block opens with one comment pointing at its help.

In [ ]:
show(start, Line((2, 3), (12, 40), color=8))

Two lines, one comment.

In [ ]:
show(start, Composite((Line((2, 3), (12, 40), color=8), Line((50, 60), (30, 45), color=11))))

## Cells written out: `AsciiGrid` (cost 8 + one per cell)

A rectangle written as a picture in the notation `ascii_grid()` prints, read back by `from_ascii_grid`. Its cost grows with its area, which is what keeps it for content nothing else describes.

In [ ]:
show(start, AsciiGrid(4, 20, ("ee0ee", "e080e", "ee0ee")))

## Real changes: `Composite`

Parts apply in order; the cost is their sum.

### tu93, level 1, the first `RIGHT`

The player tile moved six columns right, the tile it left turned white, and the energy bar lost its last pixel. Nineteen cells changed; as single cells that is 76, as a move and a cell 14.

In [ ]:
first_move = Composite(
    (
        BlockCopy(15, 18, 21, 24, src_row=15, src_col=15, clear=0),
        SetPixel(63, 63, 0),
    )
)
show(frames["level1_start"], first_move, frames["level1_after_right"])

### tu93, level 2, the `RIGHT` that lost

The player stepped right into the square in front of a red guard, and the guard slid six columns left onto it: the red tile landed where the player went, its indicator turned yellow, the player's old tile and the guard's old tile went white, and the bar lost a pixel. The model spent some 24,000 characters of reasoning establishing this turn from pixels. As a move of the guard, a fill of the player's old tile and two cells it is 24; the decomposer finds a shorter reading at 18, the whole strip shifted six columns.

In [ ]:
guard_slide = Composite(
    (
        BlockCopy(27, 30, 30, 33, src_row=27, src_col=36, clear=0),
        SetPixel(28, 30, 11),
        FillRect(27, 30, 24, 27, color=0),
        SetPixel(63, 58, 0),
    )
)
show(frames["level2_before_right"], guard_slide, frames["level2_after_right"])

## The names the block uses

The block is written against whatever expressions the model holds the frames under. `a` and `b` may be names or chains of attributes, subscripts and calls; `g` is a plain name the block assigns.

In [ ]:
print(guard_slide.to_code(a="history[-2].current_frame.grid", b="current_frame.grid", g="g"))

## What the self-test refuses

`to_code()` runs its own text on probe frames, under `block_namespace()`, before returning it. A transform whose text does not do what `transform` does never emits, and neither does one whose text writes to its start frame.

In [ ]:
class Lying(SetPixel):
    """Writes a different colour than it sets."""

    def _code_lines(self, a, g):
        return [f"{g}[{self.row}, {self.col}] = {(self.color + 1) % 16}"]


try:
    Lying(5, 5, 3).to_code()
except AssertionError as error:
    print(error)

A rectangle outside the frame is refused at construction; numpy would otherwise clip it silently, in process and in the block alike.

In [ ]:
try:
    FillRect(60, 66, 0, 5, color=1)
except ValueError as error:
    print(error)

## Costs

The costs are declared, not derived: they set an order. A 3×3 sprite that moved is cheapest as a copy, a horizontal run of one colour is a fill rather than a line, and a single cell is a single cell.

In [ ]:
sprite = start[15:18, 15:18]
for label, transform in [
    ("3x3 sprite as a copy", BlockCopy(40, 43, 40, 43, src_row=15, src_col=15)),
    ("3x3 sprite as a move", BlockCopy(40, 43, 40, 43, src_row=15, src_col=15, clear=0)),
    ("3x3 sprite as a paste", AsciiGrid.from_array(40, 40, sprite)),
    (
        "3x3 sprite as cells",
        Composite(tuple(SetPixel(40 + r, 40 + c, int(sprite[r, c])) for r in range(3) for c in range(3))),
    ),
    ("horizontal run as a fill", FillRect(10, 11, 5, 25, color=8)),
    ("horizontal run as a line", Line((10, 5), (10, 24), color=8)),
]:
    print(f"{label:28} {transform.bit_cost():5g}")